# Creating Sets in Python — Advanced Tutorial Problems with Solutions

This notebook uses a **tutorial-first style**.

Instead of jumping directly to finished answers, each problem is divided into logical stages:

1. Understand the requirement
2. Predict Python's behavior
3. Build a partial result
4. Improve the construction
5. Test edge cases
6. Extract a reusable rule

The topic is set creation, but the deeper questions are:

- What counts as the same value?
- Should values be normalized first?
- Are the elements hashable?
- Does order matter?
- Can the input be consumed more than once?
- Is a normal `set` or a `frozenset` the better model?

All examples use only the Python standard library.

## Setup

The notebook uses assertions as lightweight tests.  
Set display order may differ between Python runs.

In [1]:
from __future__ import annotations

from collections import Counter
from collections.abc import Hashable, Iterable
from dataclasses import dataclass
from itertools import chain
from typing import Any, TypeVar
import math
import re

T = TypeVar("T")

# Part 1 — Syntax and Identity

## Problem 1 — Empty Set or Empty Dictionary?

A common mistake is to write:

```python
values = {}
```

and expect an empty set.

Before running the next cell, predict the type of each expression:

```python
{}
set()
{1}
{"x": 1}
```

In [2]:
examples = [
    {},
    set(),
    {1},
    {"x": 1},
]

for value in examples:
    print(f"{value!r:12} -> {type(value).__name__}")

{}           -> dict
set()        -> set
{1}          -> set
{'x': 1}     -> dict


### Explanation

Python uses curly braces for both dictionaries and sets.

- `{key: value}` creates a dictionary.
- `{value1, value2}` creates a set.
- `{}` is reserved for an empty dictionary.
- `set()` is required for an empty set.

In [3]:
empty_set = set()
empty_dictionary = {}

assert type(empty_set) is set
assert type(empty_dictionary) is dict

### Solution 1

Use `set()` whenever no initial elements are available.

This pattern is especially common in incremental algorithms:

```python
seen = set()
seen.add("first")
```

## Problem 2 — Equality Can Collapse Different Types

Consider:

```python
{1, 1.0, True, "1", (1,), None}
```

There are six expressions, but how many set elements remain?

In [4]:
mixed_values = {
    1,
    1.0,
    True,
    "1",
    (1,),
    None,
}

print(mixed_values)
print("element count:", len(mixed_values))

{None, 1, (1,), '1'}
element count: 4


### Step 1 — Inspect Equality

Sets use equality and hashing to determine uniqueness.

In Python, these values compare equal:

```python
1 == 1.0 == True
```

In [5]:
assert 1 == 1.0
assert 1 == True
assert hash(1) == hash(1.0) == hash(True)

### Step 2 — Identify the Distinct Groups

The values form four distinct groups:

- `1`, `1.0`, and `True`
- `"1"`
- `(1,)`
- `None`

In [6]:
assert len(mixed_values) == 4
assert 1 in mixed_values
assert "1" in mixed_values
assert (1,) in mixed_values
assert None in mixed_values

### Solution 2

Python's equality rules determine set identity.

If a domain must distinguish booleans from integer IDs, validate the values before inserting them.

# Part 2 — Normalization before Deduplication

## Problem 3 — Canonical Usernames

A registration system receives noisy usernames:

```python
[
    "  Alice ",
    "ALICE",
    "bob",
    " Bob ",
    "",
    "   ",
    "carol",
]
```

Create a set of canonical usernames.

Rules:

1. Remove outer whitespace.
2. Compare case-insensitively.
3. Ignore blanks.

In [7]:
raw_usernames = [
    "  Alice ",
    "ALICE",
    "bob",
    " Bob ",
    "",
    "   ",
    "carol",
]

### Step 1 — Try Direct Set Conversion

Direct conversion removes only exact duplicates.

In [8]:
direct_result = set(raw_usernames)
print(direct_result)

{'ALICE', '', '   ', '  Alice ', 'carol', 'bob', ' Bob '}


The direct result still contains multiple representations of the same logical username.

We need a canonical form before deduplication.

### Step 2 — Define the Canonical Form

In [9]:
def canonical_username(value: str) -> str:
    return value.strip().casefold()


for value in raw_usernames:
    print(
        f"{value!r:12} -> "
        f"{canonical_username(value)!r}"
    )

'  Alice '   -> 'alice'
'ALICE'      -> 'alice'
'bob'        -> 'bob'
' Bob '      -> 'bob'
''           -> ''
'   '        -> ''
'carol'      -> 'carol'


### Step 3 — Create the Set

A set comprehension performs transformation, filtering, and deduplication.

In [10]:
usernames = {
    normalized
    for value in raw_usernames
    if (normalized := canonical_username(value))
}

assert usernames == {
    "alice",
    "bob",
    "carol",
}

print(usernames)

{'carol', 'alice', 'bob'}


### Step 4 — Test Useful Properties

A good canonicalization pipeline should be:

- idempotent,
- unaffected by duplicate input,
- unaffected by input order.

In [11]:
assert {
    canonical_username(value)
    for value in usernames
} == usernames

assert {
    canonical_username(value)
    for value in raw_usernames + raw_usernames
    if canonical_username(value)
} == usernames

assert {
    canonical_username(value)
    for value in reversed(raw_usernames)
    if canonical_username(value)
} == usernames

### Solution 3

Normalize **before** creating the set.

The set cannot infer that differently formatted strings represent the same domain value.

## Problem 4 — Canonical Product Codes

Product codes should:

- ignore surrounding whitespace,
- use uppercase,
- remove internal hyphens,
- contain only letters and digits,
- ignore invalid blank values.

In [12]:
raw_codes = [
    " ab-100 ",
    "AB100",
    "xy-200",
    "XY200",
    "",
    "bad code",
]

### Step 1 — Normalize One Code

In [13]:
def canonical_code(value: str) -> str:
    cleaned = value.strip().upper().replace("-", "")

    if cleaned and cleaned.isalnum():
        return cleaned

    return ""


for value in raw_codes:
    print(
        f"{value!r:12} -> "
        f"{canonical_code(value)!r}"
    )

' ab-100 '   -> 'AB100'
'AB100'      -> 'AB100'
'xy-200'     -> 'XY200'
'XY200'      -> 'XY200'
''           -> ''
'bad code'   -> ''


### Step 2 — Build the Distinct Code Set

In [14]:
product_codes = {
    normalized
    for value in raw_codes
    if (normalized := canonical_code(value))
}

assert product_codes == {
    "AB100",
    "XY200",
}

print(product_codes)

{'AB100', 'XY200'}


### Solution 4

Set construction is often the final stage of a data-cleaning pipeline:

```python
raw value -> validation -> normalization -> set insertion
```

# Part 3 — Creating Sets from Mappings and Records

## Problem 5 — Dictionary Keys, Values, and Items

A dictionary maps course names to instructors.

Create:

1. A set of course names
2. A set of instructors
3. A set of `(course, instructor)` assignments

In [15]:
course_instructors = {
    "Python": "Maya",
    "Databases": "Leo",
    "Algorithms": "Maya",
}

### Step 1 — Remember Dictionary Iteration

Iterating a dictionary directly yields keys.

In [16]:
for item in course_instructors:
    print(item)

Python
Databases
Algorithms


### Step 2 — Construct the Three Sets

In [17]:
courses = set(course_instructors)
instructors = set(course_instructors.values())
assignments = set(course_instructors.items())

assert courses == {
    "Python",
    "Databases",
    "Algorithms",
}
assert instructors == {
    "Maya",
    "Leo",
}
assert assignments == {
    ("Python", "Maya"),
    ("Databases", "Leo"),
    ("Algorithms", "Maya"),
}

print("courses:", courses)
print("instructors:", instructors)
print("assignments:", assignments)

courses: {'Algorithms', 'Databases', 'Python'}
instructors: {'Leo', 'Maya'}
assignments: {('Algorithms', 'Maya'), ('Python', 'Maya'), ('Databases', 'Leo')}


### Step 3 — Understand the Hashability Requirement

`set(mapping.items())` works only when every key and value is hashable.

If a dictionary value is a list, convert it first.

In [18]:
course_topics = {
    "Python": ["syntax", "sets"],
    "Databases": ["sql", "indexes"],
}

hashable_topics = {
    (course, tuple(topics))
    for course, topics in course_topics.items()
}

assert hashable_topics == {
    ("Python", ("syntax", "sets")),
    ("Databases", ("sql", "indexes")),
}

### Solution 5

Use:

- `set(mapping)` for keys,
- `set(mapping.values())` for values,
- `set(mapping.items())` for hashable key-value pairs.

## Problem 6 — Customers with Large Purchases

Each transaction is `(customer, amount)`.

Create a set of customers who made at least one purchase of 100 or more.

In [19]:
transactions = [
    ("Ava", 20),
    ("Noah", 120),
    ("Ava", 180),
    ("Mia", 75),
    ("Noah", 200),
    ("Liam", 100),
]

### Step 1 — Express the Rule

For every transaction:

- keep it if the amount is at least 100,
- extract the customer,
- let the set remove repeated customer names.

In [20]:
large_purchase_customers = {
    customer
    for customer, amount in transactions
    if amount >= 100
}

assert large_purchase_customers == {
    "Ava",
    "Noah",
    "Liam",
}

print(large_purchase_customers)

{'Noah', 'Liam', 'Ava'}


### Step 2 — Compare with a List

A list keeps repeated qualifying customers.

In [21]:
large_purchase_list = [
    customer
    for customer, amount in transactions
    if amount >= 100
]

print(large_purchase_list)
assert large_purchase_list.count("Noah") == 2

['Noah', 'Ava', 'Noah', 'Liam']


### Solution 6

Use a set comprehension when the final requirement is a distinct transformed or filtered collection.

# Part 4 — Combining Iterables

## Problem 7 — Merge Several ID Sources

Three systems provide IDs through different iterable types:

- list,
- tuple,
- generator.

Create one set of all distinct IDs.

In [22]:
legacy_ids = [10, 11, 12, 12]
current_ids = (12, 13, 14)
imported_ids = (
    value
    for value in [14, 15, 16]
)

### Step 1 — Merge Fixed Sources with Unpacking

In [23]:
fixed_merge = {
    *legacy_ids,
    *current_ids,
}

assert fixed_merge == {
    10, 11, 12, 13, 14
}

### Step 2 — Include the Generator

In [24]:
all_ids = {
    *legacy_ids,
    *current_ids,
    *imported_ids,
}

assert all_ids == {
    10, 11, 12, 13, 14, 15, 16
}

print(all_ids)

{10, 11, 12, 13, 14, 15, 16}


### Step 3 — Observe Generator Exhaustion

The generator has already been consumed.

In [25]:
assert set(imported_ids) == set()

### Step 4 — Merge a Dynamic Collection of Sources

For a dynamic number of iterables, use `chain.from_iterable`.

In [26]:
sources = [
    [1, 2, 3],
    (3, 4),
    range(4, 7),
]

dynamic_merge = set(
    chain.from_iterable(sources)
)

assert dynamic_merge == {
    1, 2, 3, 4, 5, 6
}

### Solution 7

Use unpacking for a small fixed number of iterables.  
Use `chain.from_iterable` when the source collection is dynamic.

## Problem 8 — Summarize a One-Shot Stream

A reading generator must be processed once.

Return:

1. The set of distinct readings
2. The total number of readings

In [27]:
def reading_stream():
    yield from [
        18,
        19,
        18,
        20,
        21,
        20,
    ]

### Step 1 — Avoid Repeated Iteration

A generator cannot be restarted after consumption.

Instead, calculate both outputs in one pass.

In [28]:
def summarize_stream(values: Iterable[T]):
    distinct: set[T] = set()
    count = 0

    for value in values:
        distinct.add(value)
        count += 1

    return distinct, count


distinct_readings, reading_count = summarize_stream(
    reading_stream()
)

assert distinct_readings == {
    18, 19, 20, 21
}
assert reading_count == 6

### Step 2 — Compare with Materialization

If repeated access to every original reading is required, create a list snapshot deliberately.

In [29]:
snapshot = list(reading_stream())

assert set(snapshot) == distinct_readings
assert len(snapshot) == reading_count

### Solution 8

For one-shot inputs:

- summarize in one pass when only aggregates are needed,
- materialize only when repeated access is genuinely required.

# Part 5 — Hashable Representations

## Problem 9 — Deduplicate Coordinate Paths

Each path is a list of coordinate lists.

The complete path must become a set element.

Because lists are unhashable, every nested list must be converted.

In [30]:
raw_paths = [
    [[0, 0], [1, 0], [1, 1]],
    [[0, 0], [1, 0], [1, 1]],
    [[0, 0], [0, 1], [1, 1]],
]

### Step 1 — Convert One Coordinate

In [31]:
coordinate = [4, 7]
hashable_coordinate = tuple(coordinate)

assert hashable_coordinate == (4, 7)
hash(hashable_coordinate)

-3793500297614796835

### Step 2 — Convert One Full Path

In [32]:
first_path = tuple(
    tuple(point)
    for point in raw_paths[0]
)

assert first_path == (
    (0, 0),
    (1, 0),
    (1, 1),
)

hash(first_path)

-645496946838686438

### Step 3 — Build the Set of Paths

In [33]:
unique_paths = {
    tuple(
        tuple(point)
        for point in path
    )
    for path in raw_paths
}

assert len(unique_paths) == 2
print(unique_paths)

{((0, 0), (1, 0), (1, 1)), ((0, 0), (0, 1), (1, 1))}


### Solution 9

When order matters, recursively convert list-like sequences to tuples.

## Problem 10 — Deduplicate Unordered Teams

A team is a collection of member names.

Requirements:

- member order does not matter,
- repeated names inside a team do not matter,
- duplicate teams should collapse.

In [34]:
raw_teams = [
    ["Ava", "Noah", "Mia"],
    ["Mia", "Ava", "Noah"],
    ["Liam", "Emma"],
    ["Emma", "Liam", "Emma"],
]

### Step 1 — See Why Tuples Are Not Ideal

Tuple order affects equality.

In [35]:
assert tuple(raw_teams[0]) != tuple(raw_teams[1])

### Step 2 — Use `frozenset`

`frozenset` is immutable, hashable, and unordered.

In [36]:
team_a = frozenset(raw_teams[0])
team_b = frozenset(raw_teams[1])

assert team_a == team_b
hash(team_a)

6586717106860787839

### Step 3 — Build the Unique Team Set

In [37]:
unique_teams = {
    frozenset(team)
    for team in raw_teams
}

assert unique_teams == {
    frozenset({
        "Ava",
        "Noah",
        "Mia",
    }),
    frozenset({
        "Liam",
        "Emma",
    }),
}

print(unique_teams)

{frozenset({'Emma', 'Liam'}), frozenset({'Noah', 'Mia', 'Ava'})}


### Solution 10

Use:

- tuple when element order matters,
- `frozenset` when only membership matters.

## Problem 11 — Recursively Freeze Nested Records

Configuration records contain dictionaries, lists, tuples, and sets.

Convert them into stable hashable representations.

In [38]:
configurations = [
    {
        "mode": "fast",
        "retries": [1, 2],
        "flags": {"cache", "log"},
    },
    {
        "flags": {"log", "cache"},
        "retries": [1, 2],
        "mode": "fast",
    },
    {
        "mode": "safe",
        "retries": [1],
        "flags": {"log"},
    },
]

### Step 1 — Define the Rules

- dictionary → sorted tuple of frozen pairs
- list → tuple
- tuple → tuple of frozen items
- set → `frozenset`
- scalar → unchanged if hashable

In [39]:
def freeze(value: Any) -> Hashable:
    if isinstance(value, dict):
        frozen_pairs = (
            (
                freeze(key),
                freeze(item_value),
            )
            for key, item_value in value.items()
        )
        return tuple(
            sorted(
                frozen_pairs,
                key=repr,
            )
        )

    if isinstance(value, list):
        return tuple(
            freeze(item)
            for item in value
        )

    if isinstance(value, tuple):
        return tuple(
            freeze(item)
            for item in value
        )

    if isinstance(value, set):
        return frozenset(
            freeze(item)
            for item in value
        )

    hash(value)
    return value

### Step 2 — Freeze and Deduplicate

In [40]:
frozen_first = freeze(configurations[0])
hash(frozen_first)

unique_configurations = {
    freeze(configuration)
    for configuration in configurations
}

assert len(unique_configurations) == 2
print(unique_configurations)

{(('flags', frozenset({'cache', 'log'})), ('mode', 'fast'), ('retries', (1, 2))), (('flags', frozenset({'log'})), ('mode', 'safe'), ('retries', (1,)))}


### Explanation

The function preserves list order but ignores set order.

That behavior is a deliberate canonicalization policy, not a universal rule.

### Solution 11

Nested values can become set elements only after the complete representation is hashable.

# Part 6 — Sets inside Larger Algorithms

## Problem 12 — Preserve First Appearance

Remove duplicates from:

```python
["B", "A", "B", "C", "A", "D"]
```

while preserving the order of first appearance.

A set alone cannot represent that order.

In [41]:
codes = [
    "B",
    "A",
    "B",
    "C",
    "A",
    "D",
]

### Step 1 — Use a Set for Membership

Maintain:

- a `seen` set for fast membership checks,
- a result list for ordered output.

In [42]:
def unique_in_order(values: Iterable[T]) -> list[T]:
    seen: set[T] = set()
    result: list[T] = []

    for value in values:
        if value not in seen:
            seen.add(value)
            result.append(value)

    return result


ordered_codes = unique_in_order(codes)

assert ordered_codes == [
    "B",
    "A",
    "C",
    "D",
]

### Step 2 — Compare with `dict.fromkeys`

In [43]:
assert list(
    dict.fromkeys(codes)
) == ordered_codes

### Solution 12

A set can support an ordered algorithm without being the final output type.

## Problem 13 — Deduplicate Products by SKU

Two product records are considered the same when their normalized SKUs match.

Other fields do not determine identity.

In [44]:
products = [
    {
        "sku": " ab-100 ",
        "name": "Keyboard",
        "price": 40,
    },
    {
        "sku": "AB-100",
        "name": "Mechanical Keyboard",
        "price": 55,
    },
    {
        "sku": "xy-200",
        "name": "Mouse",
        "price": 20,
    },
]

### Step 1 — Define the Identity Projection

In [45]:
def canonical_sku(value: str) -> str:
    return value.strip().upper()


assert canonical_sku(
    products[0]["sku"]
) == canonical_sku(
    products[1]["sku"]
)

### Step 2 — Keep the First Record for Each Identity

In [46]:
def unique_products_by_sku(records):
    seen_skus = set()
    result = []

    for record in records:
        sku = canonical_sku(record["sku"])

        if not sku:
            raise ValueError(
                "SKU cannot be blank"
            )

        if sku not in seen_skus:
            seen_skus.add(sku)
            result.append(record)

    return result


unique_products = unique_products_by_sku(
    products
)

assert len(unique_products) == 2
assert unique_products[0]["name"] == "Keyboard"
assert unique_products[1]["name"] == "Mouse"

### Step 3 — Create Only the Identity Set

In [47]:
unique_skus = {
    canonical_sku(
        product["sku"]
    )
    for product in products
}

assert unique_skus == {
    "AB-100",
    "XY-200",
}

### Solution 13

Sets have no custom `key=` parameter.

When identity depends on one field or a transformation, track the derived key explicitly.

# Part 7 — Validation and Special Values

## Problem 14 — Positive Integer IDs Only

The domain requires positive integer IDs.

Reject:

- booleans,
- strings,
- zero,
- negative numbers,
- unhashable values.

In [48]:
raw_ids = [
    10,
    20,
    True,
    "30",
    10,
    -5,
    [40],
]

### Step 1 — Define Domain Validity

`bool` must be excluded explicitly because it is a subclass of `int`.

In [49]:
def is_valid_id(value: Any) -> bool:
    return (
        isinstance(value, int)
        and not isinstance(value, bool)
        and value > 0
    )


for value in raw_ids:
    print(
        f"{value!r:8} -> "
        f"{is_valid_id(value)}"
    )

10       -> True
20       -> True
True     -> False
'30'     -> False
10       -> True
-5       -> False
[40]     -> False


### Step 2 — Filtering Version

In [50]:
valid_ids = {
    value
    for value in raw_ids
    if is_valid_id(value)
}

assert valid_ids == {
    10,
    20,
}

### Step 3 — Strict Version

In [51]:
def strict_id_set(
    values: Iterable[Any],
) -> set[int]:
    result: set[int] = set()

    for index, value in enumerate(values):
        if not is_valid_id(value):
            raise TypeError(
                f"Invalid ID at index "
                f"{index}: {value!r}"
            )

        result.add(value)

    return result


try:
    strict_id_set(raw_ids)
except TypeError as exc:
    print(exc)
else:
    raise AssertionError(
        "Expected TypeError"
    )

Invalid ID at index 2: True


### Solution 14

Hashability is not the same as domain validity.

Choose explicitly between filtering invalid data and rejecting the entire batch.

## Problem 15 — Finite Measurements Only

Create a set containing only finite measurements.

The input includes duplicate values, infinity, and multiple `NaN` objects.

In [52]:
measurements = [
    1.0,
    2.5,
    float("nan"),
    1.0,
    float("inf"),
    -3.0,
    float("nan"),
]

### Step 1 — Understand `NaN`

`NaN` does not compare equal to itself.

In [53]:
nan_value = float("nan")
assert nan_value != nan_value

nan_a = float("nan")
nan_b = float("nan")
nan_set = {
    nan_a,
    nan_b,
}

assert len(nan_set) == 2
print("NaN set length:", len(nan_set))

NaN set length: 2


### Step 2 — Filter with `math.isfinite`

In [54]:
finite_measurements = {
    value
    for value in measurements
    if math.isfinite(value)
}

assert finite_measurements == {
    1.0,
    2.5,
    -3.0,
}

print(finite_measurements)

{1.0, 2.5, -3.0}


### Solution 15

Sets enforce uniqueness according to Python equality, but they do not automatically clean special numeric values.

# Part 8 — Advanced Set Elements

## Problem 16 — Distinct Bigrams

A bigram is a pair of consecutive words.

Create the set of distinct bigrams in a sentence.

In [55]:
sentence = (
    "Sets remove duplicates and sets "
    "support fast membership checks"
)

### Step 1 — Tokenize

In [56]:
tokens = re.findall(
    r"[a-z]+",
    sentence.casefold(),
)

print(tokens)

['sets', 'remove', 'duplicates', 'and', 'sets', 'support', 'fast', 'membership', 'checks']


### Step 2 — Construct Tuple Elements

Each bigram is an ordered pair, so a tuple is appropriate.

In [57]:
bigrams = {
    (
        tokens[index],
        tokens[index + 1],
    )
    for index in range(
        len(tokens) - 1
    )
}

assert bigrams == {
    ("sets", "remove"),
    ("remove", "duplicates"),
    ("duplicates", "and"),
    ("and", "sets"),
    ("sets", "support"),
    ("support", "fast"),
    ("fast", "membership"),
    ("membership", "checks"),
}

print(bigrams)

{('sets', 'remove'), ('and', 'sets'), ('fast', 'membership'), ('duplicates', 'and'), ('remove', 'duplicates'), ('support', 'fast'), ('sets', 'support'), ('membership', 'checks')}


### Solution 16

Use tuples as set elements when order inside each record is meaningful.

## Problem 17 — Distinct Unordered Relationships

Friendship is symmetric:

```python
("Ava", "Noah")
```

is the same as:

```python
("Noah", "Ava")
```

Ignore self-friendships.

In [58]:
friendship_records = [
    ("Ava", "Noah"),
    ("Noah", "Ava"),
    ("Mia", "Liam"),
    ("Liam", "Mia"),
    ("Ava", "Ava"),
]

### Step 1 — Use `frozenset` for Symmetry

In [59]:
friendships = {
    frozenset(
        (person_a, person_b)
    )
    for person_a, person_b in friendship_records
    if person_a != person_b
}

assert friendships == {
    frozenset({
        "Ava",
        "Noah",
    }),
    frozenset({
        "Mia",
        "Liam",
    }),
}

print(friendships)

{frozenset({'Mia', 'Liam'}), frozenset({'Noah', 'Ava'})}


### Explanation

A tuple models an ordered relation.  
A `frozenset` models immutable unordered membership.

### Solution 17

Choose the hashable element type according to whether internal order is meaningful.

## Problem 18 — Distinct Letter-Frequency Signatures

Words such as `"listen"` and `"silent"` have the same letter counts.

Create a set of distinct letter-frequency signatures.

In [60]:
signature_words = [
    "listen",
    "silent",
    "enlist",
    "google",
    "gogole",
    "python",
]

### Step 1 — Build a Frequency Mapping

In [61]:
example_counts = Counter("listen")
print(example_counts)

Counter({'l': 1, 'i': 1, 's': 1, 't': 1, 'e': 1, 'n': 1})


A `Counter` is mutable and unhashable.

Convert it to a stable sorted tuple of pairs.

In [62]:
def letter_signature(word: str):
    return tuple(
        sorted(
            Counter(
                word.casefold()
            ).items()
        )
    )


assert letter_signature(
    "listen"
) == letter_signature(
    "silent"
)

hash(letter_signature("listen"))

2878715408701321900

### Step 2 — Create the Signature Set

In [63]:
signatures = {
    letter_signature(word)
    for word in signature_words
}

assert len(signatures) == 3
print(signatures)

{(('e', 1), ('i', 1), ('l', 1), ('n', 1), ('s', 1), ('t', 1)), (('e', 1), ('g', 2), ('l', 1), ('o', 2)), (('h', 1), ('n', 1), ('o', 1), ('p', 1), ('t', 1), ('y', 1))}


### Solution 18

Complex domain identities can be inserted into sets after conversion to stable hashable representations.

# Part 9 — Capstone Tutorial

## Problem 19 — Build a Course Skill Taxonomy

A course catalog contains noisy skill labels and repeated course records.

Create:

1. A set of all canonical skills
2. A mapping from each skill to the set of course IDs teaching it
3. A set of skills taught by more than one course
4. A set of unique skill bundles, where skill order does not matter

In [64]:
course_catalog = [
    {
        "course_id": "PY101",
        "skills": [
            " Python ",
            "sets",
            "Hashing",
            "sets",
        ],
    },
    {
        "course_id": "DS201",
        "skills": [
            "python",
            " Data Cleaning ",
            "HASHING",
        ],
    },
    {
        "course_id": "DB301",
        "skills": [
            "SQL",
            "indexes",
            "",
        ],
    },
    {
        "course_id": "PY101",
        "skills": [
            "Python",
            "SETS",
            "hashing",
        ],
    },
]

### Step 1 — Canonicalize Skill Labels

Rules:

- trim whitespace,
- casefold,
- collapse internal whitespace,
- replace spaces with hyphens,
- ignore blanks.

In [65]:
def canonical_skill(value: str) -> str:
    words = (
        value
        .strip()
        .casefold()
        .split()
    )
    return "-".join(words)


assert canonical_skill(
    " Data Cleaning "
) == "data-cleaning"
assert canonical_skill("   ") == ""

### Step 2 — Normalize One Course

A set removes repeated skills within the same course.

In [66]:
first_course_skills = {
    normalized
    for skill in course_catalog[0]["skills"]
    if (
        normalized
        := canonical_skill(skill)
    )
}

assert first_course_skills == {
    "python",
    "sets",
    "hashing",
}

### Step 3 — Build the Global Skill Set

In [67]:
all_skills = {
    normalized
    for course in course_catalog
    for skill in course["skills"]
    if (
        normalized
        := canonical_skill(skill)
    )
}

assert all_skills == {
    "python",
    "sets",
    "hashing",
    "data-cleaning",
    "sql",
    "indexes",
}

print(all_skills)

{'python', 'hashing', 'sets', 'sql', 'data-cleaning', 'indexes'}


### Step 4 — Build the Skill-to-Course Index

Each dictionary value is a set of distinct course IDs.

In [68]:
skill_to_courses = {}

for course in course_catalog:
    course_id = (
        course["course_id"]
        .strip()
        .upper()
    )

    normalized_skills = {
        normalized
        for skill in course["skills"]
        if (
            normalized
            := canonical_skill(skill)
        )
    }

    for skill in normalized_skills:
        skill_to_courses.setdefault(
            skill,
            set(),
        ).add(course_id)


for skill in sorted(skill_to_courses):
    print(
        f"{skill:14} -> "
        f"{sorted(skill_to_courses[skill])}"
    )

data-cleaning  -> ['DS201']
hashing        -> ['DS201', 'PY101']
indexes        -> ['DB301']
python         -> ['DS201', 'PY101']
sets           -> ['PY101']
sql            -> ['DB301']


### Step 5 — Test the Index

In [69]:
assert skill_to_courses["python"] == {
    "PY101",
    "DS201",
}
assert skill_to_courses["sets"] == {
    "PY101",
}
assert skill_to_courses["hashing"] == {
    "PY101",
    "DS201",
}

### Step 6 — Find Shared Skills

In [70]:
shared_skills = {
    skill
    for skill, course_ids
    in skill_to_courses.items()
    if len(course_ids) > 1
}

assert shared_skills == {
    "python",
    "hashing",
}

print(shared_skills)

{'python', 'hashing'}


### Step 7 — Create Unique Skill Bundles

Each bundle is unordered and must itself be hashable.

In [71]:
skill_bundles = {
    frozenset(
        normalized
        for skill in course["skills"]
        if (
            normalized
            := canonical_skill(skill)
        )
    )
    for course in course_catalog
}

assert skill_bundles == {
    frozenset({
        "python",
        "sets",
        "hashing",
    }),
    frozenset({
        "python",
        "data-cleaning",
        "hashing",
    }),
    frozenset({
        "sql",
        "indexes",
    }),
}

print(skill_bundles)

{frozenset({'sql', 'indexes'}), frozenset({'python', 'hashing', 'sets'}), frozenset({'python', 'data-cleaning', 'hashing'})}


### Step 8 — Extract Reusable Functions

In [72]:
def course_skill_set(
    skills: Iterable[str],
) -> set[str]:
    return {
        normalized
        for skill in skills
        if (
            normalized
            := canonical_skill(skill)
        )
    }


def build_skill_index(
    catalog: Iterable[dict[str, Any]],
) -> dict[str, set[str]]:
    index: dict[str, set[str]] = {}

    for course in catalog:
        course_id = (
            str(course["course_id"])
            .strip()
            .upper()
        )

        if not course_id:
            raise ValueError(
                "course_id cannot be blank"
            )

        for skill in course_skill_set(
            course["skills"]
        ):
            index.setdefault(
                skill,
                set(),
            ).add(course_id)

    return index


rebuilt_index = build_skill_index(
    course_catalog
)

assert rebuilt_index == skill_to_courses

### Solution 19

The capstone combines several set-creation techniques:

- normalized set comprehensions,
- nested loops that update sets stored in a dictionary,
- `frozenset` for immutable unordered bundles,
- set filtering based on cardinality.

The most important step is defining identity before constructing the set.

# Review Exercises

## Review 1 — Email Domains

Create a set of canonical domains from:

```python
[
    "A@Example.com",
    "b@example.COM",
    "c@school.edu",
    "invalid",
]
```

Ignore entries without an `@`.

In [73]:
email_values = [
    "A@Example.com",
    "b@example.COM",
    "c@school.edu",
    "invalid",
]

domains = {
    value.rsplit("@", 1)[1].casefold()
    for value in email_values
    if "@" in value
}

assert domains == {
    "example.com",
    "school.edu",
}

## Review 2 — Canonical Integer Groups

Convert:

```python
[[3, 1], [1, 3], [2, 2]]
```

to a set of distinct sorted tuples.

In [74]:
number_groups = [
    [3, 1],
    [1, 3],
    [2, 2],
]

canonical_groups = {
    tuple(sorted(group))
    for group in number_groups
}

assert canonical_groups == {
    (1, 3),
    (2, 2),
}

## Review 3 — Words Appearing in Multiple Documents

Create a set of words that occur in at least two separate documents.

In [75]:
review_documents = [
    "sets are useful",
    "useful tools include sets",
    "lists preserve order",
]

document_word_sets = [
    set(
        re.findall(
            r"[a-z]+",
            document.casefold(),
        )
    )
    for document in review_documents
]

word_document_counts = Counter(
    word
    for word_set in document_word_sets
    for word in word_set
)

words_in_multiple_documents = {
    word
    for word, count
    in word_document_counts.items()
    if count >= 2
}

assert words_in_multiple_documents == {
    "sets",
    "useful",
}

## Review 4 — Permission Bundles

Create a set of unique immutable permission bundles.

In [76]:
role_permissions = {
    "viewer": ["read"],
    "editor": ["read", "write"],
    "author": ["write", "read"],
}

permission_bundles = {
    frozenset(permissions)
    for permissions
    in role_permissions.values()
}

assert permission_bundles == {
    frozenset({"read"}),
    frozenset({
        "read",
        "write",
    }),
}

## Review 5 — Event Stream Summary

Process an event iterator once and return:

- the set of event types,
- the total event count.

In [77]:
def summarize_events(events):
    event_types = set()
    event_count = 0

    for event in events:
        event_types.add(
            event["type"]
        )
        event_count += 1

    return event_types, event_count


event_stream = (
    event
    for event in [
        {"type": "login"},
        {"type": "purchase"},
        {"type": "login"},
    ]
)

event_types, event_count = summarize_events(
    event_stream
)

assert event_types == {
    "login",
    "purchase",
}
assert event_count == 3

# Final Mental Model

Before creating a set, answer these questions.

## 1. What counts as the same value?

Python equality may be sufficient, or a canonical form may be required.

## 2. Are the elements hashable?

Lists, dictionaries, and mutable sets need a hashable representation.

## 3. Does order matter?

- Use a set as the final result when order does not matter.
- Use a set internally when order must be preserved elsewhere.

## 4. Is the input reusable?

Generators and streams are often one-shot.

## 5. Should invalid data be filtered or rejected?

Make the policy explicit.

## 6. Can the set be constructed directly?

Prefer a set comprehension over an unnecessary intermediate list.

# Compact Reference

```python
# Empty set
result = set()

# Convert an iterable
result = set(iterable)

# Transform
result = {
    transform(item)
    for item in iterable
}

# Filter
result = {
    item
    for item in iterable
    if predicate(item)
}

# Normalize and filter
result = {
    normalized
    for item in iterable
    if (
        normalized
        := normalize(item)
    )
}

# Merge fixed iterables
result = {
    *first,
    *second,
    *third,
}

# Merge dynamic iterables
result = set(
    chain.from_iterable(iterables)
)

# Immutable unordered groups
result = {
    frozenset(group)
    for group in groups
}

# Preserve first appearance
seen = set()
ordered = []

for item in iterable:
    if item not in seen:
        seen.add(item)
        ordered.append(item)
```